In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import mean_squared_error

# Function to load data from a CSV file and handle missing Brand_Name values
def recsys_products_to_clients_load_data(path):
    data = pd.read_csv(path)
    data['Brand_Name'].fillna('Unknown', inplace=True)
    return data

# Function to encode Client_ID and Product_ID using LabelEncoder
def recsys_products_to_clients_encode_ids(data):
    client_encoder = LabelEncoder()
    product_encoder = LabelEncoder()
    data['Client_ID'] = client_encoder.fit_transform(data['Client_ID'])
    data['Product_ID'] = product_encoder.fit_transform(data['Product_ID'])
    return data, client_encoder, product_encoder

# Function to aggregate duplicate entries by summing rating values
def recsys_products_to_clients_aggregate_duplicates(data, client_col, product_col, rating_col):
    aggregated_data = data.groupby([client_col, product_col], as_index=False)[rating_col].sum()
    aggregated_data = aggregated_data.merge(data[['Product_ID', 'Product_Name']].drop_duplicates(), on=product_col, how='left')
    return aggregated_data

# Function to create an interaction matrix (client-product rating matrix)
def recsys_products_to_clients_create_interaction_matrix(data, client_col, product_col, rating_col):
    interaction_matrix = data.pivot(index=client_col, columns=product_col, values=rating_col).fillna(0)
    return interaction_matrix

# Function to build a matrix factorization model using TensorFlow and Keras
def recsys_products_to_clients_build_matrix_factorization_model(num_clients, num_products, embedding_dim=25, l2_reg=1e-6, dropout_rate=0.5):
    client_input = keras.layers.Input(shape=(1,), name='client_input')
    client_embedding = keras.layers.Embedding(input_dim=num_clients, output_dim=embedding_dim,
                                              embeddings_regularizer=keras.regularizers.l2(l2_reg),
                                              name='client_embedding')(client_input)
    client_vec = keras.layers.Flatten()(client_embedding)

    product_input = keras.layers.Input(shape=(1,), name='product_input')
    product_embedding = keras.layers.Embedding(input_dim=num_products, output_dim=embedding_dim,
                                               embeddings_regularizer=keras.regularizers.l2(l2_reg),
                                               name='product_embedding')(product_input)
    product_vec = keras.layers.Flatten()(product_embedding)

    concatenated = keras.layers.Concatenate()([client_vec, product_vec])
    dropout = keras.layers.Dropout(dropout_rate)(concatenated)
    output = keras.layers.Dense(1)(dropout)

    model = keras.models.Model(inputs=[client_input, product_input], outputs=output)
    model.compile(optimizer='adam', loss='mse')
    return model

# Function to train the matrix factorization model
def recsys_products_to_clients_train_matrix_factorization_model(model, X_train, y_train, epochs=5, batch_size=64, validation_split=0.2):
    model.fit([X_train[:, 0], X_train[:, 1]], y_train, epochs=epochs, batch_size=batch_size, validation_split=validation_split)

# Function to generate product recommendations for all clients
def recsys_products_to_clients_generate_recommendations_for_all_clients(data, client_encoder, product_encoder, model, num_products):
    recommendations = []

    # Iterate over all clients
    for client_id in range(len(client_encoder.classes_)):
        client_product_combinations = np.array([(client_id, product) for product in range(num_products)])

        all_client_ids = client_product_combinations[:, 0]
        all_product_ids = client_product_combinations[:, 1]

        predictions = model.predict([all_client_ids, all_product_ids]).flatten()
        scaler = MinMaxScaler(feature_range=(0, 100))
        predictions_normalized = scaler.fit_transform(predictions.reshape(-1, 1)).flatten()

        client_recommendations = pd.DataFrame({
            'Client_ID': client_encoder.inverse_transform([client_id] * len(predictions_normalized)),
            'Product_ID': product_encoder.inverse_transform(all_product_ids),
            'Predicted_Rating': predictions_normalized
        })

        recommendations.append(client_recommendations)

    # Concatenate all client recommendations into a single DataFrame
    recommendations_df = pd.concat(recommendations, ignore_index=True)
    return recommendations_df

# Function to compute MSE for all recommendations
def compute_mse_for_recommendations(model, X, y):
    y_pred = model.predict([X[:, 0], X[:, 1]]).flatten()
    mse = mean_squared_error(y, y_pred)
    return mse

# Main execution
if __name__ == "__main__":
    data_path = "/Users/igormol/Desktop/Analytics_NC2/sample/CPP.csv"
    data = recsys_products_to_clients_load_data(data_path)
    data, client_encoder, product_encoder = recsys_products_to_clients_encode_ids(data)

    aggregated_data = recsys_products_to_clients_aggregate_duplicates(data, 'Client_ID', 'Product_ID', 'Rating')
    interaction_matrix = recsys_products_to_clients_create_interaction_matrix(aggregated_data, 'Client_ID', 'Product_ID', 'Rating')

    num_clients = interaction_matrix.shape[0]
    num_products = interaction_matrix.shape[1]

    X = aggregated_data[['Client_ID', 'Product_ID']].values
    y = aggregated_data['Rating'].values

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = recsys_products_to_clients_build_matrix_factorization_model(num_clients, num_products)
    recsys_products_to_clients_train_matrix_factorization_model(model, X_train, y_train)

    recommendations_df = recsys_products_to_clients_generate_recommendations_for_all_clients(data, client_encoder, product_encoder, model, num_products)
    
    # Save the recommendations to a CSV file
    recommendations_df.to_csv("/Users/igormol/Desktop/Analytics_NC2/sample/recommendations.csv", index=False)


In [ ]:
# Function to load data from a CSV file and handle missing Brand_Name values
def recsys_products_to_clients_load_data(path):
    data = pd.read_csv(path)
    data['Brand_Name'].fillna('Unknown', inplace=True)
    print("Columns in the dataset:", data.columns)  # Print column names for debugging
    return data

# Main execution
if __name__ == "__main__":
    data_path = "/Users/igormol/Desktop/Analytics_NC2/sample/CPP.csv"
    data = recsys_products_to_clients_load_data(data_path)
    
    # Use the correct column name instead of 'Rating'
    rating_column = 'Actual_Rating_Column_Name'  # Replace with the correct column name
    data, client_encoder, product_encoder = recsys_products_to_clients_encode_ids(data)

    aggregated_data = recsys_products_to_clients_aggregate_duplicates(data, 'Client_ID', 'Product_ID', rating_column)
    interaction_matrix = recsys_products_to_clients_create_interaction_matrix(aggregated_data, 'Client_ID', 'Product_ID', rating_column)

    num_clients = interaction_matrix.shape[0]
    num_products = interaction_matrix.shape[1]

    X = aggregated_data[['Client_ID', 'Product_ID']].values
    y = aggregated_data[rating_column].values

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = recsys_products_to_clients_build_matrix_factorization_model(num_clients, num_products)
    recsys_products_to_clients_train_matrix_factorization_model(model, X_train, y_train)

    recommendations_df = recsys_products_to_clients_generate_recommendations_for_all_clients(data, client_encoder, product_encoder, model, num_products)
    
    # Save the recommendations to a CSV file
    recommendations_df.to_csv("/Users/igormol/Desktop/Analytics_NC2/sample/recommendations.csv", index=False)


In [ ]:
# recsys_products_clients.py

# Makes 2024 Copyright
# Igor Mol <igor.mol@makes.ai>

import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

# Function to load data from a CSV file and handle missing Brand_Name values
def recsys_products_to_clients_load_data(path):
    data = pd.read_csv(path)
    data['Brand_Name'].fillna('Unknown', inplace=True)
    return data

# Function to encode Client_ID and Product_ID using LabelEncoder
def recsys_products_to_clients_encode_ids(data):
    client_encoder = LabelEncoder()
    product_encoder = LabelEncoder()

    data['Client_ID'] = client_encoder.fit_transform(data['Client_ID'])
    data['Product_ID'] = product_encoder.fit_transform(data['Product_ID'])
    return data, client_encoder, product_encoder

# Function to create an interaction matrix (client-product matrix)
def recsys_products_to_clients_create_interaction_matrix(data, client_col, product_col):
    interaction_matrix = data.pivot(index=client_col, columns=product_col, values=product_col).notnull().astype(int)
    return interaction_matrix

# Function to build a matrix factorization model using TensorFlow and Keras
def recsys_products_to_clients_build_matrix_factorization_model(num_clients, num_products, embedding_dim=25, l2_reg=1e-6, dropout_rate=0.5):
    client_input = keras.layers.Input(shape=(1,), name='client_input')
    client_embedding = keras.layers.Embedding(input_dim=num_clients, output_dim=embedding_dim,
                                              embeddings_regularizer=keras.regularizers.l2(l2_reg),
                                              name='client_embedding')(client_input)
    client_vec = keras.layers.Flatten()(client_embedding)

    product_input = keras.layers.Input(shape=(1,), name='product_input')
    product_embedding = keras.layers.Embedding(input_dim=num_products, output_dim=embedding_dim,
                                               embeddings_regularizer=keras.regularizers.l2(l2_reg),
                                               name='product_embedding')(product_input)
    product_vec = keras.layers.Flatten()(product_embedding)

    concatenated = keras.layers.Concatenate()([client_vec, product_vec])
    dropout = keras.layers.Dropout(dropout_rate)(concatenated)
    output = keras.layers.Dense(1)(dropout)

    model = keras.models.Model(inputs=[client_input, product_input], outputs=output)
    model.compile(optimizer='adam', loss='mse')
    return model

# Function to train the matrix factorization model
def recsys_products_to_clients_train_matrix_factorization_model(model, X_train, y_train, epochs=5, batch_size=64, validation_split=0.2):
    model.fit([X_train[:, 0], X_train[:, 1]], y_train, epochs=epochs, batch_size=batch_size, validation_split=validation_split)

# Function to generate recommendations for all clients
def recsys_products_to_clients_generate_recommendations_for_all_clients(client_encoder, product_encoder, model, num_clients, num_products):
    recommendations = []

    for client_id in range(num_clients):
        client_product_combinations = np.array([(client_id, product) for product in range(num_products)])
        all_client_ids = client_product_combinations[:, 0]
        all_product_ids = client_product_combinations[:, 1]
        predictions = model.predict([all_client_ids, all_product_ids]).flatten()

        recommendations.append(pd.DataFrame({
            'Client_ID': client_encoder.inverse_transform([client_id] * num_products),
            'Product_ID': product_encoder.inverse_transform(all_product_ids),
            'Predicted_Score': predictions
        }))

    return pd.concat(recommendations, ignore_index=True)

# Main function to execute the recommendation generation and save to CSV
def main():
    data_path = "/Users/igormol/Desktop/Analytics_NC2/sample/CPP.csv"
    data = recsys_products_to_clients_load_data(data_path)
    data, client_encoder, product_encoder = recsys_products_to_clients_encode_ids(data)
    
    num_clients = len(client_encoder.classes_)
    num_products = len(product_encoder.classes_)

    interaction_matrix = recsys_products_to_clients_create_interaction_matrix(data, 'Client_ID', 'Product_ID')

    model = recsys_products_to_clients_build_matrix_factorization_model(num_clients, num_products)
    X = interaction_matrix.stack().reset_index().values[:, :2]
    y = interaction_matrix.stack().values

    recsys_products_to_clients_train_matrix_factorization_model(model, X, y)

    recommendations = recsys_products_to_clients_generate_recommendations_for_all_clients(client_encoder, product_encoder, model, num_clients, num_products)
    
    # Save the recommendations to a CSV file
    recommendations.to_csv("/Users/igormol/Desktop/Analytics_NC2/sample/CPP_recommendations.csv", index=False)

if __name__ == "__main__":
    main()


In [ ]:
# recsys_products_clients.py

# Makes 2024 Copyright
# Igor Mol <igor.mol@makes.ai>

import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

# Function to load data from a CSV file and handle missing Brand_Name values
def recsys_products_to_clients_load_data(path):
    data = pd.read_csv(path)
    data['Brand_Name'].fillna('Unknown', inplace=True)
    return data

# Function to encode Client_ID and Product_ID using LabelEncoder
def recsys_products_to_clients_encode_ids(data):
    client_encoder = LabelEncoder()
    product_encoder = LabelEncoder()

    data['Client_ID'] = client_encoder.fit_transform(data['Client_ID'])
    data['Product_ID'] = product_encoder.fit_transform(data['Product_ID'])
    return data, client_encoder, product_encoder

# Function to aggregate duplicate entries by summing occurrences
def recsys_products_to_clients_aggregate_duplicates(data, client_col, product_col):
    # Group data by client and product, counting occurrences
    aggregated_data = data.groupby([client_col, product_col], as_index=False).size()
    aggregated_data = aggregated_data.rename(columns={'size': 'Interaction'})
    return aggregated_data

# Function to create an interaction matrix (client-product interaction matrix)
def recsys_products_to_clients_create_interaction_matrix(aggregated_data, client_col, product_col, interaction_col):
    # Pivot the data to create the interaction matrix
    interaction_matrix = aggregated_data.pivot(index=client_col, columns=product_col, values=interaction_col).fillna(0)
    return interaction_matrix

# Function to build a matrix factorization model using TensorFlow and Keras
def recsys_products_to_clients_build_matrix_factorization_model(num_clients, num_products, embedding_dim=25, l2_reg=1e-6, dropout_rate=0.5):
    client_input = keras.layers.Input(shape=(1,), name='client_input')
    client_embedding = keras.layers.Embedding(input_dim=num_clients, output_dim=embedding_dim,
                                              embeddings_regularizer=keras.regularizers.l2(l2_reg),
                                              name='client_embedding')(client_input)
    client_vec = keras.layers.Flatten()(client_embedding)

    product_input = keras.layers.Input(shape=(1,), name='product_input')
    product_embedding = keras.layers.Embedding(input_dim=num_products, output_dim=embedding_dim,
                                               embeddings_regularizer=keras.regularizers.l2(l2_reg),
                                               name='product_embedding')(product_input)
    product_vec = keras.layers.Flatten()(product_embedding)

    concatenated = keras.layers.Concatenate()([client_vec, product_vec])
    dropout = keras.layers.Dropout(dropout_rate)(concatenated)
    output = keras.layers.Dense(1)(dropout)

    model = keras.models.Model(inputs=[client_input, product_input], outputs=output)
    model.compile(optimizer='adam', loss='mse')
    return model

# Function to train the matrix factorization model
def recsys_products_to_clients_train_matrix_factorization_model(model, X_train, y_train, epochs=5, batch_size=64, validation_split=0.2):
    model.fit([X_train[:, 0], X_train[:, 1]], y_train, epochs=epochs, batch_size=batch_size, validation_split=validation_split)

# Function to generate recommendations for all clients
def recsys_products_to_clients_generate_recommendations_for_all_clients(client_encoder, product_encoder, model, num_clients, num_products):
    recommendations = []

    for client_id in range(num_clients):
        client_product_combinations = np.array([(client_id, product) for product in range(num_products)])
        all_client_ids = client_product_combinations[:, 0]
        all_product_ids = client_product_combinations[:, 1]
        predictions = model.predict([all_client_ids, all_product_ids]).flatten()

        recommendations.append(pd.DataFrame({
            'Client_ID': client_encoder.inverse_transform([client_id] * num_products),
            'Product_ID': product_encoder.inverse_transform(all_product_ids),
            'Predicted_Score': predictions
        }))

    return pd.concat(recommendations, ignore_index=True)

# Main function to execute the recommendation generation and save to CSV
def main():
    data_path = "/Users/igormol/Desktop/Analytics_NC2/sample/CPP.csv"
    data = recsys_products_to_clients_load_data(data_path)
    data, client_encoder, product_encoder = recsys_products_to_clients_encode_ids(data)

    aggregated_data = recsys_products_to_clients_aggregate_duplicates(data, 'Client_ID', 'Product_ID')

    num_clients = len(client_encoder.classes_)
    num_products = len(product_encoder.classes_)

    interaction_matrix = recsys_products_to_clients_create_interaction_matrix(aggregated_data, 'Client_ID', 'Product_ID', 'Interaction')

    model = recsys_products_to_clients_build_matrix_factorization_model(num_clients, num_products)
    X = interaction_matrix.stack().reset_index().values[:, :2]
    y = interaction_matrix.stack().values

    recsys_products_to_clients_train_matrix_factorization_model(model, X, y)

    recommendations = recsys_products_to_clients_generate_recommendations_for_all_clients(client_encoder, product_encoder, model, num_clients, num_products)

    # Save the recommendations to a CSV file
    recommendations.to_csv("/Users/igormol/Desktop/Analytics_NC2/sample/CPP_recommendations.csv", index=False)

if __name__ == "__main__":
    main()
